# Lab 12 — Transfer Learning com MobileNetV3

## Objetivos
1. Aplicar Transfer Learning com MobileNetV3-Small pré-treinado no ImageNet
2. Comparar Feature Extraction vs Fine-Tuning no CIFAR-10
3. Comparar com o resultado do Lab 10 (CNN treinada do zero)
4. Salvar o modelo treinado para deploy via FastAPI

---

**Dataset:** CIFAR-10 (mesmo do Lab 10, para comparação direta)

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device}')

Usando: cpu


## Parte 1 — Dataset CIFAR-10 com pré-processamento para ImageNet

In [2]:
CLASSES = ['aviao', 'automovel', 'passaro', 'gato', 'cervo',
           'cachorro', 'sapo', 'cavalo', 'navio', 'caminhao']

# Redimensiona para 224x224 (tamanho esperado pelo MobileNetV3)
# Normalização ImageNet para aproveitar os pesos pré-treinados
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2470, 0.2435, 0.2616]),
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2470, 0.2435, 0.2616]),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=True, transform=transform_val)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=64, shuffle=False, num_workers=2)

print(f'Treino: {len(trainset)} amostras | Teste: {len(testset)} amostras')

100.0%


Treino: 50000 amostras | Teste: 10000 amostras


## Parte 2 — Construindo o modelo de Transfer Learning

In [3]:
def criar_modelo(freeze_backbone: bool = True, num_classes: int = 10):
    """Carrega MobileNetV3-Small e substitui o classifier para num_classes."""
    model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
        print('Backbone congelado — apenas o classifier será treinado')
    else:
        print('Fine-tuning completo — backbone + classifier serão treinados')

    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)

    treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Parâmetros treináveis: {treinaveis:,} / {total:,} ({100*treinaveis/total:.1f}%)')

    return model.to(device)


# Experimento 1: Feature Extraction (backbone congelado)
model = criar_modelo(freeze_backbone=True)

Backbone congelado — apenas o classifier será treinado
Parâmetros treináveis: 601,098 / 1,528,106 (39.3%)


## Parte 3 — Treinamento

In [4]:
def treinar(model, trainloader, testloader, epochs=10, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    historico = {'train_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        # --- treino ---
        model.train()
        total_loss, corretos, total = 0.0, 0, 0
        for imgs, labels in trainloader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            preds = model(imgs)
            loss = criterion(preds, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            corretos += (preds.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * corretos / total

        # --- validação ---
        model.eval()
        corretos_val, total_val = 0, 0
        with torch.no_grad():
            for imgs, labels in testloader:
                imgs, labels = imgs.to(device), labels.to(device)
                preds = model(imgs)
                corretos_val += (preds.argmax(1) == labels).sum().item()
                total_val += labels.size(0)

        val_acc = 100 * corretos_val / total_val
        scheduler.step()

        historico['train_loss'].append(total_loss / len(trainloader))
        historico['train_acc'].append(train_acc)
        historico['val_acc'].append(val_acc)

        print(f'Época {epoch+1:02d}/{epochs} | Loss: {total_loss/len(trainloader):.3f} | '
              f'Treino: {train_acc:.1f}% | Val: {val_acc:.1f}%')

    return historico


historico = treinar(model, trainloader, testloader, epochs=10)

Época 01/10 | Loss: 0.630 | Treino: 78.2% | Val: 81.1%
Época 02/10 | Loss: 0.490 | Treino: 83.0% | Val: 84.7%
Época 03/10 | Loss: 0.437 | Treino: 84.8% | Val: 85.6%
Época 04/10 | Loss: 0.401 | Treino: 86.1% | Val: 85.9%
Época 05/10 | Loss: 0.367 | Treino: 87.0% | Val: 86.1%
Época 06/10 | Loss: 0.302 | Treino: 89.3% | Val: 86.8%
Época 07/10 | Loss: 0.284 | Treino: 90.0% | Val: 87.3%


KeyboardInterrupt: 

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(historico['train_loss'], label='Loss treino')
ax1.set_title('Loss por época'); ax1.set_xlabel('Época'); ax1.legend()
ax2.plot(historico['train_acc'], label='Acurácia treino')
ax2.plot(historico['val_acc'],   label='Acurácia validação')
ax2.set_title('Acurácia por época'); ax2.set_xlabel('Época'); ax2.legend()
plt.tight_layout()
plt.show()

print(f'\nMelhor acurácia de validação: {max(historico["val_acc"]):.1f}%')

## Parte 4 — Fine-Tuning (backbone descongelado)

Agora treinamos novamente, desta vez com o backbone também sendo atualizado (com lr menor).

In [ ]:
model_ft = criar_modelo(freeze_backbone=False)

# lr diferenciado: backbone com lr menor para não destruir os pesos pré-treinados
criterion = nn.CrossEntropyLoss()
optimizer_ft = torch.optim.Adam([
    {'params': model_ft.features.parameters(), 'lr': 1e-5},
    {'params': model_ft.classifier.parameters(), 'lr': 1e-3},
])

historico_ft = treinar(model_ft, trainloader, testloader, epochs=10)
print(f'\nMelhor acurácia fine-tuning: {max(historico_ft["val_acc"]):.1f}%')

In [ ]:
# Comparação entre as estratégias
print('=== Comparação ===')
print(f'Feature Extraction: {max(historico["val_acc"]):.1f}%')
print(f'Fine-Tuning:        {max(historico_ft["val_acc"]):.1f}%')
print('Lab 10 (CNN do zero): ~70-75%  (referência)')

plt.figure(figsize=(8, 4))
plt.plot(historico['val_acc'],    label='Feature Extraction')
plt.plot(historico_ft['val_acc'], label='Fine-Tuning')
plt.axhline(y=73, color='gray', linestyle='--', label='CNN do zero (ref.)')
plt.title('Comparação de estratégias — CIFAR-10')
plt.xlabel('Época'); plt.ylabel('Acurácia (%)')
plt.legend(); plt.tight_layout(); plt.show()

## Parte 5 — Salvar artefato para deploy

In [ ]:
# Salva o melhor modelo (fine-tuning)
artifacts_dir = Path('artifacts')
artifacts_dir.mkdir(exist_ok=True)

torch.save(model_ft.backbone.state_dict(), artifacts_dir / 'mobilenet_cifar10.pt')

metadata = {
    'model_name': 'MobileNetV3-Small (Transfer Learning)',
    'dataset': 'CIFAR-10',
    'input_size': 224,
    'num_classes': 10,
    'classes': CLASSES,
    'normalization': {
        'mean': [0.4914, 0.4822, 0.4465],
        'std':  [0.2470, 0.2435, 0.2616],
    },
    'val_acc': round(max(historico_ft['val_acc']), 2),
}
(artifacts_dir / 'metadata.json').write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8'
)

print('Salvo em artifacts/:')
for f in sorted(artifacts_dir.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.0f} KB)')

---
## Exercícios

1. **Troque o backbone:** substitua `mobilenet_v3_small` por `resnet18`. Compare acurácia e tempo de treino.

2. **Dataset próprio:** aplique transfer learning em um dataset de sua escolha com 3–5 classes. Use `torchvision.datasets.ImageFolder` para carregar imagens organizadas em pastas.

3. **Análise de erros:** gere a matriz de confusão e identifique quais classes o modelo mais confunde.

4. **Deploy:** copie `artifacts/` para a pasta da API e inicie o servidor. Teste com imagens via Postman e interface web.